In [1]:
import os
if 'experiments' in os.getcwd ():
    os.chdir (os.getcwd () + "/..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings ('ignore', category = ConvergenceWarning)

In [2]:
####### DATA PATHS #######
TRAIN_PATH = "./data/in/cattle_data_train.csv"
TEST_PATH = "./data/in/cattle_data_test.csv"
OUT_PATH = "./data/out/nnet.csv"

# Preprocessing

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler, MinMaxScaler

pd.set_option ("display.max_columns", None)
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID",
    "Farm_ID",
    "Feed_Quantity_lb",
    "Climate_Zone",
    "Management_System",
    "Feed_Type",
    "Feeding_Frequency",
    "Walking_Distance_km",
    "Grazing_Duration_hrs",
    "Rumination_Time_hrs",
    "Resting_Hours",
    "Humidity_percent",
    "BVD_Vaccine",
    "FMD_Vaccine",
    "Brucellosis_Vaccine",
    "HS_Vaccine",
    "BQ_Vaccine",
    "Housing_Score",
    "Body_Condition_Score",
    "Milking_Interval_hrs",
    "Breed",

]

CATEGORICAL_FEATURES = [
    "Date",
    # "Milking_Interval_hrs",
    "Young",
    "Lactation_Stage",

    # "IBR_Vaccine",
    # "Anthrax_Vaccine",
    # "Rabies_Vaccine"
]

STANDARD_SCALED_FEATURES = [
    "Feed_Quantity_kg",
    "Water_Intake_L",
    "Parity",
    "Ambient_Temperature_C",
    "Previous_Week_Avg_Yield",
    "Days_in_Milk",
    "Age_Months",
    "Weight_kg",
]

In [4]:
def preprocess (
    dtrain, dtest
):
    """
    NOTES:
    - Interaction features do not help
    - Squaring feed for outlier overexageratting does not help
    - clipping negative records worsens results
    - Predictive and mean imputation worsen results
    - Robust & min-max scalers worsen results
    """
    # Convert month to season
    def month_to_season (m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    months = pd.to_datetime (dtest['Date']).dt.month
    dtest = dtest.drop (columns = ['Date'])
    dtest['Date'] = months.apply (month_to_season)

    months = pd.to_datetime (dtrain['Date']).dt.month
    dtrain = dtrain.drop (columns = ['Date'])
    dtrain['Date'] = months.apply (month_to_season)

    # Breed
    dtrain['Young'] = (dtrain['Age_Months'] < 60).astype(int)
    dtest['Young'] = (dtest['Age_Months'] < 60).astype(int)

    # Imputation  
    median_val = dtrain["Feed_Quantity_kg"].median ()

    dtrain.loc[dtrain["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna (), "Feed_Quantity_kg"] = median_val

    # Drop features deemed unnecessary
    dtrain = dtrain.drop (DROP_FEATURES, axis = 1)
    dtest = dtest.drop (DROP_FEATURES, axis = 1)

    # One-hot encode
    dtrain = pd.get_dummies (dtrain, columns = CATEGORICAL_FEATURES, 
                             drop_first = True)
    dtest = pd.get_dummies (dtest, columns = CATEGORICAL_FEATURES, 
                            drop_first = True)
    dtrain, dtest = dtrain.align (dtest, join = 'left', axis = 1, fill_value = 0)

    # Standardize data
    scaler = StandardScaler ()
    dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform (
                                                dtrain[STANDARD_SCALED_FEATURES])
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform (
                                                dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

In [5]:
# NOTE: Dropping negative records worsens results
train_data = pd.read_csv (TRAIN_PATH)

X_train, X_test, y_train, y_test = train_test_split (
    train_data.drop (TARGET_FEATURE, axis = 1),
    train_data[TARGET_FEATURE], test_size = 0.2, random_state = 0)

X_train, X_test, scaler = preprocess (X_train, X_test)
# print (X_train.head ())

# Training Set Eval

In [6]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.base import clone

# RMSE at each checkpoint
train_rmse_list = []
test_rmse_list = []

total_iterations = 0
iterations_per_step = 10
model_template = MLPRegressor (hidden_layer_sizes = (110, 110, 110),
                               activation = "tanh",
                               learning_rate_init = 0.00003,
                               learning_rate = "adaptive",
                               # alpha = 0.01,
                               early_stopping = False, 
                               # validation_fraction = 0.15,
                               n_iter_no_change = 20,
                               verbose = False,
                               warm_start = True,
                               max_iter = iterations_per_step,
                               random_state = 1)

In [7]:
model = clone (model_template)
prev_rmse = float ('inf')
test_rmse = 0
TOLERANCE = 0.00005
print (model)
# total_iterations < 200
while (test_rmse + TOLERANCE < prev_rmse):
    if test_rmse > 0:
        prev_rmse = test_rmse
    model.fit (X_train, y_train)

    # Compute RMSE
    y_train_pred = model.predict (X_train)
    y_test_pred = model.predict (X_test)
    train_rmse = np.sqrt (mean_squared_error (y_train, y_train_pred))
    test_rmse = np.sqrt (mean_squared_error (y_test, y_test_pred))

    train_rmse_list.append (train_rmse)
    test_rmse_list.append (test_rmse)

    total_iterations += iterations_per_step
    print (f"Iteration {total_iterations}: Train RMSE={train_rmse:.5f}, Test RMSE={test_rmse:.5f}")

print ("Termination Condition Reached:")
print (f"Prev: {prev_rmse}")
print (f"Last: {test_rmse}", flush = True)


MLPRegressor(activation='tanh', hidden_layer_sizes=(110, 110, 110),
             learning_rate='adaptive', learning_rate_init=3e-05, max_iter=10,
             n_iter_no_change=20, random_state=1, warm_start=True)


Iteration 10: Train RMSE=4.15700, Test RMSE=4.15477
Iteration 20: Train RMSE=4.12119, Test RMSE=4.11678
Iteration 30: Train RMSE=4.11313, Test RMSE=4.10892
Iteration 40: Train RMSE=4.10959, Test RMSE=4.10607
Iteration 50: Train RMSE=4.10744, Test RMSE=4.10461
Iteration 60: Train RMSE=4.10596, Test RMSE=4.10379
Iteration 70: Train RMSE=4.10487, Test RMSE=4.10332
Iteration 80: Train RMSE=4.10401, Test RMSE=4.10308
Iteration 90: Train RMSE=4.10329, Test RMSE=4.10296
Iteration 100: Train RMSE=4.10266, Test RMSE=4.10292
Termination Condition Reached:
Prev: 4.102960149636064
Last: 4.102922196838657


# Final Model

In [8]:
# Build final model
train_data = pd.read_csv (TRAIN_PATH)
test_data = pd.read_csv (TEST_PATH)

X_train = train_data.drop (TARGET_FEATURE, axis = 1)
y_train = train_data[TARGET_FEATURE]

X_test = test_data
X_train, X_test, scaler = preprocess (X_train, X_test)

model = clone (model_template)
model.max_iter = total_iterations
model.fit (X_train, y_train)

,loss,'squared_error'
,hidden_layer_sizes,"(110, ...)"
,activation,'tanh'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'adaptive'
,learning_rate_init,3e-05
,power_t,0.5
,max_iter,100
,shuffle,True


In [10]:
# Final Predictions
y_pred = model.predict (X_test)

print (f"Training mean: {y_train.mean ()}")
print (f"Predicted mean: {y_pred.mean ()}")
out_data = pd.DataFrame ({'Cattle_ID': np.arange (1, len (y_pred) + 1),
                          'Milk_Yield_L': y_pred})
out_data.to_csv (OUT_PATH, index = False)

Training mean: 15.589155594329181
Predicted mean: 15.56692620780774
